# Tugas 1 — Milestone 3: Analisis Temporal, Spasial, dan Advanced EDA

**Mata Kuliah:** Analisis Big Data
**Target Check-in:** Pertemuan 8 (Mulai dikerjakan Pertemuan 6)
**Nama Mahasiswa:** Dino Alfian Zamri
**NIM:** 202310370311329
**Username GitHub:** Alfian79
**Dataset:** `ID_REG_KG_2511` (peraturan perundang-undangan Indonesia beserta fitur knowledge graph)

---

## Panduan Pengerjaan Milestone 3:
1. **Sumber Data:** dataset bersih hasil Milestone 2 (`data/processed/dataset_bersih/`, 731.632 baris).
2. **Analisis Temporal / Windowing:** Polars (`rolling_mean`, `shift`) dan DuckDB window function (`OVER (...)`).
3. **Visualisasi Interaktif:** 7 visualisasi Plotly.
4. **Perumusan Insight:** 5 insight yang angkanya dihitung langsung dari data.

> **Catatan unit analisis:** satu baris dataset adalah *potongan* (pasal/bagian) dari sebuah peraturan, bukan satu peraturan utuh. Karena itu analisis menghitung dua hal: jumlah potongan (`jumlah_pasal`) dan jumlah peraturan unik (`jumlah_regulasi`).

## Bagian 1: Inisialisasi dan Memuat Data Bersih

Data bersih berukuran 2,84 GB karena memuat vektor *embedding*. Untuk analisis, kolom vektor, JSON, dan teks panjang tidak diperlukan. Sel berikut membuat **tabel ringkas** (satu kali, disimpan sebagai Parquet) berisi kolom yang dipakai saja, lalu seluruh analisis memakai tabel itu.

In [ ]:
import gc
from pathlib import Path

import duckdb
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
import polars as pl
from IPython.display import Markdown, display

# Jika grafik tidak tampil di JupyterLab, hapus tanda # pada dua baris berikut:
# import plotly.io as pio
# pio.renderers.default = 'iframe'

pl.Config.set_tbl_rows(30)
pl.Config.set_fmt_str_lengths(60)


def cari_root(mulai):
    """Naik dari folder saat ini sampai menemukan akar repo (berisi requirements.txt dan Dockerfile)."""
    for p in [mulai, *mulai.parents]:
        if (p / 'requirements.txt').exists() and (p / 'Dockerfile').exists():
            return p
    return mulai


PROJECT_ROOT = cari_root(Path.cwd().resolve())
BERSIH_DIR = PROJECT_ROOT / 'data' / 'processed' / 'dataset_bersih'
RINGKAS_PATH = PROJECT_ROOT / 'data' / 'processed' / 'dataset_ringkas.parquet'
FIG_DIR = PROJECT_ROOT / 'output' / 'figures'
FIG_DIR.mkdir(parents=True, exist_ok=True)

berkas_bersih = sorted(BERSIH_DIR.glob('*.parquet'))
if not berkas_bersih:
    raise FileNotFoundError(f'Data bersih tidak ditemukan di {BERSIH_DIR}. Jalankan notebook 02 sampai selesai.')

BANGUN_ULANG = False   # True = buat ulang tabel ringkas (mis. setelah notebook 02 dijalankan ulang)

KOLOM_RINGAN = [
    'global_id', 'regulation_type', 'enacting_body', 'regulation_number', 'year', 'effective_date',
    'article', 'chunk_id', 'kg_primary_domain', 'kg_domain_confidence', 'kg_entity_count',
    'kg_cross_ref_count', 'kg_cluster_count', 'kg_cluster_diversity', 'kg_authority_score',
    'kg_hierarchy_level', 'kg_temporal_score', 'kg_years_old', 'kg_legal_richness',
    'kg_legal_complexity', 'kg_has_obligations', 'kg_has_prohibitions', 'kg_has_permissions',
    'kg_completeness_score', 'kg_connectivity_score', 'kg_entity_count_outlier',
    'kg_cross_ref_count_outlier', 'year_diperbaiki',
]

if BANGUN_ULANG or not RINGKAS_PATH.exists():
    potongan = []
    for i, f in enumerate(berkas_bersih, start=1):
        potongan.append(
            pl.scan_parquet(f)
            .select(KOLOM_RINGAN + [pl.col('content').str.len_chars().alias('panjang_content')])
            .collect()
        )
        if i % 15 == 0 or i == len(berkas_bersih):
            print(f'Tabel ringkas: {i}/{len(berkas_bersih)} berkas')
    pl.concat(potongan).write_parquet(RINGKAS_PATH, compression='zstd')
    del potongan
    gc.collect()

df = pl.read_parquet(RINGKAS_PATH)
print(f'Data dimuat: {df.height:,} baris x {df.width} kolom')
print(f'Ukuran tabel ringkas di memori: {df.estimated_size() / 1e6:.0f} MB')

In [ ]:
# --- Kolom turunan -------------------------------------------------------------------------
# 1) Normalisasi variasi penulisan jenis peraturan (temuan Milestone 2)
NORMALISASI_JENIS = {
    'UNDANG UNDANG': 'UNDANG-UNDANG', 'UU': 'UNDANG-UNDANG', 'UUD': 'UNDANG-UNDANG DASAR',
    'PP': 'PERATURAN PEMERINTAH', 'KEPPRES': 'KEPUTUSAN PRESIDEN', 'PERDA': 'PERATURAN DAERAH',
    'PERBUP': 'PERATURAN BUPATI', 'SE': 'SURAT EDARAN', 'PERPU': 'PERATURAN PEMERINTAH PENGGANTI UNDANG-UNDANG',
    'PBI': 'PERATURAN BI',
}

# 2) Tingkat pemerintahan penerbit (pendekatan spasial: data tidak punya koordinat)
TINGKAT = {
    'UNDANG-UNDANG': 'Pusat', 'UNDANG-UNDANG DASAR': 'Pusat', 'PERATURAN PEMERINTAH': 'Pusat',
    'PERATURAN PEMERINTAH PENGGANTI UNDANG-UNDANG': 'Pusat', 'PERATURAN PRESIDEN': 'Pusat',
    'KEPUTUSAN PRESIDEN': 'Pusat', 'INSTRUKSI PRESIDEN': 'Pusat', 'PERATURAN MENTERI': 'Pusat',
    'PERATURAN BI': 'Pusat',
    'PERATURAN GUBERNUR': 'Provinsi', 'PERATURAN DAERAH PROVINSI': 'Provinsi', 'KEPUTUSAN GUBERNUR': 'Provinsi',
    'PERATURAN DAERAH KABUPATEN': 'Kabupaten', 'PERATURAN BUPATI': 'Kabupaten', 'KEPUTUSAN BUPATI': 'Kabupaten',
    'PERATURAN DAERAH KOTA': 'Kota', 'PERATURAN WALIKOTA': 'Kota', 'KEPUTUSAN WALIKOTA': 'Kota',
    'Tidak diketahui': 'Tidak diketahui',
}

# 3) Bulan berlaku dari `effective_date` (format "01 Agustus 2001")
BULAN = {'januari': '1', 'februari': '2', 'maret': '3', 'april': '4', 'mei': '5', 'juni': '6',
         'juli': '7', 'agustus': '8', 'september': '9', 'oktober': '10', 'november': '11', 'desember': '12'}
NAMA_BULAN = ['Jan', 'Feb', 'Mar', 'Apr', 'Mei', 'Jun', 'Jul', 'Agu', 'Sep', 'Okt', 'Nov', 'Des']

df = df.with_columns(
    pl.col('year').cast(pl.Int64, strict=False).alias('tahun'),
    pl.col('regulation_type').fill_null('Tidak diketahui').replace(NORMALISASI_JENIS).alias('jenis_regulasi'),
    pl.col('effective_date').str.extract(r'(?i)^\s*\d{1,2}\s+([a-z]+)\s+\d{4}\s*$', 1)
        .str.to_lowercase().replace(BULAN).cast(pl.Int8, strict=False).alias('bulan_berlaku'),
    pl.col('effective_date').str.extract(r'(?i)^\s*\d{1,2}\s+[a-z]+\s+(\d{4})\s*$', 1)
        .cast(pl.Int32, strict=False).alias('tahun_berlaku'),
).with_columns(
    pl.col('jenis_regulasi').replace_strict(TINGKAT, default='Lainnya').alias('tingkat'),
    pl.concat_str(
        [pl.col(c).fill_null('?') for c in ('regulation_type', 'regulation_number', 'year', 'enacting_body')],
        separator='|',
    ).alias('kunci_regulasi'),
)

print('Jenis peraturan setelah normalisasi:', df['jenis_regulasi'].n_unique(), 'nilai')
print(f"Baris dengan tahun valid     : {df['tahun'].is_not_null().sum():,} ({df['tahun'].is_not_null().mean() * 100:.2f}%)")
print(f"Baris dengan bulan terbaca   : {df['bulan_berlaku'].is_not_null().sum():,} ({df['bulan_berlaku'].is_not_null().mean() * 100:.2f}%)")
print(f"Peraturan unik (kunci)       : {df['kunci_regulasi'].n_unique():,}")
display(df.group_by('tingkat').len().sort('len', descending=True))

## Bagian 2: Analisis Temporal dan Window Functions (Materi Pertemuan 6)

Sumbu waktu memakai `year` (`effective_date` kosong pada 34,66% baris). Dua pendekatan dipakai pada deret jumlah peraturan per tahun:

1. **Polars:** `rolling_mean` (rata-rata bergerak 3 tahun) dan `shift` (pertumbuhan tahun ke tahun).
2. **DuckDB:** *window function* `OVER (...)` untuk total kumulatif, rata-rata bergerak 5 tahun, dan peringkat tahun.

Tahun yang tidak punya data diisi 0 terlebih dahulu agar `shift` dan *window* tidak membandingkan tahun yang tidak berurutan.

In [ ]:
# --- Polars: agregasi per tahun + rolling_mean + shift ------------------------------------------
valid = df.filter(pl.col('tahun').is_not_null())
t_min, t_max = int(valid['tahun'].min()), int(valid['tahun'].max())

kalender = pl.DataFrame({'tahun': list(range(t_min, t_max + 1))}).with_columns(pl.col('tahun').cast(pl.Int64))

per_tahun = (
    kalender.join(
        valid.group_by('tahun').agg(
            pl.len().alias('jumlah_pasal'),
            pl.col('kunci_regulasi').n_unique().alias('jumlah_regulasi'),
        ),
        on='tahun', how='left',
    )
    .fill_null(0)
    .sort('tahun')
    .with_columns(
        pl.col('jumlah_regulasi').rolling_mean(window_size=3).alias('rata2_bergerak_3th'),
        pl.col('jumlah_regulasi').shift(1).alias('regulasi_tahun_lalu'),
    )
    .with_columns(
        pl.when(pl.col('regulasi_tahun_lalu') > 0)
        .then((pl.col('jumlah_regulasi') / pl.col('regulasi_tahun_lalu') - 1) * 100)
        .otherwise(None)
        .alias('pertumbuhan_yoy_persen')
    )
)
print(f'Rentang tahun: {t_min} sampai {t_max}')
display(per_tahun.tail(12))

# --- DuckDB: window function OVER (...) -----------------------------------------------------------
con = duckdb.connect()
con.register('per_tahun', per_tahun.select('tahun', 'jumlah_pasal', 'jumlah_regulasi').to_arrow())

window_sql = con.execute("""
    SELECT
        tahun,
        jumlah_regulasi,
        CAST(SUM(jumlah_regulasi) OVER (ORDER BY tahun) AS BIGINT)                          AS kumulatif_regulasi,
        AVG(jumlah_regulasi) OVER (ORDER BY tahun ROWS BETWEEN 4 PRECEDING AND CURRENT ROW) AS rata2_bergerak_5th,
        CAST(RANK() OVER (ORDER BY jumlah_regulasi DESC) AS BIGINT)                         AS peringkat_tahun
    FROM per_tahun
    ORDER BY tahun
""").pl()

print('Lima tahun dengan peraturan terbanyak (peringkat dari window function DuckDB):')
display(window_sql.sort('peringkat_tahun').head(5))

## Bagian 3: Visualisasi Interaktif (Plotly)

Seluruh grafik interaktif (hover, zoom, pilih rentang). Grafik juga disimpan sebagai HTML di `output/figures/`.

In [ ]:
# Visualisasi 1: Tren waktu (line chart + range slider)
fig1 = go.Figure()
fig1.add_trace(go.Scatter(x=per_tahun['tahun'], y=per_tahun['jumlah_regulasi'], mode='lines+markers',
                          name='Jumlah peraturan unik'))
fig1.add_trace(go.Scatter(x=per_tahun['tahun'], y=per_tahun['rata2_bergerak_3th'], mode='lines',
                          name='Rata-rata bergerak 3 tahun', line={'dash': 'dash'}))
fig1.update_layout(title='Jumlah peraturan unik per tahun (1945 sampai 2026)',
                   xaxis_title='Tahun', yaxis_title='Jumlah peraturan', hovermode='x unified')
fig1.update_xaxes(rangeslider_visible=True)
fig1.write_html(FIG_DIR / 'v1_tren_regulasi.html', include_plotlyjs='cdn')
fig1.show()

In [ ]:
# Visualisasi 2: Distribusi dan outlier panjang teks per tingkat pemerintahan (skala log)
sampel = df.sample(n=30_000, seed=42)
fig2 = px.box(sampel, x='tingkat', y='panjang_content', color='tingkat', log_y=True,
              title='Distribusi panjang teks (karakter) per tingkat pemerintahan — sampel 30.000 baris',
              labels={'panjang_content': 'Panjang teks (log)', 'tingkat': 'Tingkat'})
fig2.update_layout(showlegend=False)
fig2.write_html(FIG_DIR / 'v2_distribusi_panjang_teks.html', include_plotlyjs='cdn')
fig2.show()

In [ ]:
# Visualisasi 3: Korelasi antar-variabel numerik (heatmap)
# Catatan: kg_entity_count dan kg_cross_ref_count sudah di-winsorize pada Milestone 2.
KOLOM_KORELASI = ['kg_entity_count', 'kg_cross_ref_count', 'kg_cluster_count', 'kg_cluster_diversity',
                  'kg_authority_score', 'kg_hierarchy_level', 'kg_temporal_score', 'kg_legal_richness',
                  'kg_legal_complexity', 'kg_completeness_score', 'kg_connectivity_score', 'panjang_content']
korelasi = df.select(KOLOM_KORELASI).corr()
matriks_korelasi = korelasi.to_numpy()

fig3 = px.imshow(matriks_korelasi, x=KOLOM_KORELASI, y=KOLOM_KORELASI, zmin=-1, zmax=1,
                 color_continuous_scale='RdBu_r', text_auto='.2f', aspect='auto',
                 title='Korelasi Pearson antar-variabel numerik (seluruh 731.632 baris)')
fig3.update_layout(height=650)
fig3.write_html(FIG_DIR / 'v3_korelasi.html', include_plotlyjs='cdn')
fig3.show()

In [ ]:
# Visualisasi 4: Komposisi jenis peraturan (treemap tingkat -> jenis)
komposisi = (df.group_by('tingkat', 'jenis_regulasi').agg(pl.len().alias('jumlah_pasal'))
             .sort('jumlah_pasal', descending=True))
fig4 = px.treemap(komposisi, path=['tingkat', 'jenis_regulasi'], values='jumlah_pasal',
                  title='Komposisi potongan peraturan menurut tingkat pemerintahan dan jenis')
fig4.update_traces(textinfo='label+value+percent root')
fig4.write_html(FIG_DIR / 'v4_komposisi_jenis.html', include_plotlyjs='cdn')
fig4.show()

In [ ]:
# Visualisasi 5: Siklus musiman — bulan berlaku per dekade (heatmap)
musim = (df.filter(pl.col('bulan_berlaku').is_not_null() & pl.col('tahun_berlaku').is_between(1945, 2026))
         .with_columns((pl.col('tahun_berlaku') // 10 * 10).alias('dekade'))
         .group_by('dekade', 'bulan_berlaku').agg(pl.col('kunci_regulasi').n_unique().alias('n'))
         .select('dekade', 'bulan_berlaku', 'n'))

daftar_dekade = sorted(set(musim['dekade'].to_list()))
matriks_musim = np.zeros((len(daftar_dekade), 12), dtype=int)
for dekade, bulan, n in musim.iter_rows():
    matriks_musim[daftar_dekade.index(dekade), int(bulan) - 1] = n

fig5 = px.imshow(matriks_musim, x=NAMA_BULAN, y=[f'{d}-an' for d in daftar_dekade], aspect='auto',
                 color_continuous_scale='Viridis', text_auto=True,
                 labels={'x': 'Bulan berlaku', 'y': 'Dekade', 'color': 'Peraturan unik'},
                 title='Pola musiman: jumlah peraturan unik menurut bulan berlaku dan dekade')
fig5.write_html(FIG_DIR / 'v5_musiman_bulan_dekade.html', include_plotlyjs='cdn')
fig5.show()

In [ ]:
# Visualisasi 6: Korelasi multi-dimensi — kompleksitas vs otoritas hukum (ukuran = jumlah entitas)
sampel_scatter = df.sample(n=15_000, seed=7).with_columns((pl.col('kg_entity_count') + 1).alias('ukuran_entitas'))
fig6 = px.scatter(sampel_scatter, x='kg_legal_complexity', y='kg_authority_score', color='tingkat',
                  size='ukuran_entitas', size_max=14, opacity=0.5,
                  hover_data=['jenis_regulasi', 'tahun', 'kg_entity_count'],
                  title='Kompleksitas vs otoritas hukum menurut tingkat (sampel 15.000 baris)',
                  labels={'kg_legal_complexity': 'Kompleksitas hukum', 'kg_authority_score': 'Skor otoritas'})
fig6.write_html(FIG_DIR / 'v6_multidimensi.html', include_plotlyjs='cdn')
fig6.show()

In [ ]:
# Visualisasi 7: Pola spasial (pendekatan) — 20 lembaga penerbit dengan potongan peraturan terbanyak
# Data tidak memiliki koordinat, sehingga peta tidak dibuat; wilayah diwakili oleh `enacting_body`.
top_penerbit = (df.filter(pl.col('enacting_body').is_not_null())
                .group_by('enacting_body')
                .agg(pl.len().alias('jumlah_pasal'), pl.col('tingkat').first().alias('tingkat'))
                .sort('jumlah_pasal', descending=True).head(20))
fig7 = px.bar(top_penerbit, x='jumlah_pasal', y='enacting_body', color='tingkat', orientation='h',
              title='20 penerbit peraturan dengan potongan terbanyak',
              labels={'jumlah_pasal': 'Jumlah potongan peraturan', 'enacting_body': 'Penerbit'})
fig7.update_layout(yaxis={'categoryorder': 'total ascending'}, height=650)
fig7.write_html(FIG_DIR / 'v7_penerbit_teratas.html', include_plotlyjs='cdn')
fig7.show()

## Bagian 4: Rumusan 5 Insight Analitik Utama

Sel berikut menghitung angka-angka insight langsung dari data, sehingga setiap pernyataan bisa ditelusuri ke hasil analisis. **Bacalah hasilnya bersama grafik di atas, lalu tulis ulang dengan bahasa Anda sendiri** pada sel Markdown di bawahnya (aturan integritas akademik: Anda harus dapat menjelaskan setiap temuan).

In [ ]:
# --- Angka untuk setiap insight (semuanya dihitung dari data) ------------------------------------
# 1) Tren
baris_puncak = per_tahun.sort('jumlah_regulasi', descending=True).row(0, named=True)
total_regulasi = int(per_tahun['jumlah_regulasi'].sum())
sejak_2010 = int(per_tahun.filter(pl.col('tahun') >= 2010)['jumlah_regulasi'].sum())

# 2) Distribusi
median_teks = float(df['panjang_content'].median())
p99_teks = float(df['panjang_content'].quantile(0.99))
terpotong = int((df['panjang_content'] >= 32767).sum())
persen_entity_outlier = float(df['kg_entity_count_outlier'].mean() * 100)
persen_xref_outlier = float(df['kg_cross_ref_count_outlier'].mean() * 100)

# 3) Korelasi (pasangan terkuat, tanpa diagonal)
n = len(KOLOM_KORELASI)
pasangan = [(abs(matriks_korelasi[i, j]), matriks_korelasi[i, j], KOLOM_KORELASI[i], KOLOM_KORELASI[j])
            for i in range(n) for j in range(i + 1, n) if not np.isnan(matriks_korelasi[i, j])]
pasangan.sort(reverse=True)
_, r_kuat, var_a, var_b = pasangan[0]

# 4) Segmen
segmen = (df.group_by('tingkat').agg(
    pl.len().alias('jumlah_pasal'),
    pl.col('kg_authority_score').mean().alias('rata2_otoritas'),
    pl.col('kg_legal_complexity').mean().alias('rata2_kompleksitas'),
    pl.col('panjang_content').median().alias('median_panjang_teks'),
).sort('jumlah_pasal', descending=True))
display(segmen)
tertinggi_otoritas = segmen.sort('rata2_otoritas', descending=True).row(0, named=True)
terendah_otoritas = segmen.sort('rata2_otoritas').row(0, named=True)
terbesar = segmen.row(0, named=True)

# 5) Kualitas data / rekomendasi
persen_tanpa_tanggal = float(df['effective_date'].is_null().mean() * 100)
persen_tanpa_penerbit = float(df['enacting_body'].is_null().mean() * 100)
persen_tanpa_jenis = float((df['regulation_type'].is_null()).mean() * 100)
bulan_terbanyak = None
if musim.height:
    per_bulan = musim.group_by('bulan_berlaku').agg(pl.col('n').sum().alias('n')).sort('n', descending=True)
    bulan_terbanyak = NAMA_BULAN[int(per_bulan['bulan_berlaku'][0]) - 1]

display(Markdown(f"""
**Insight 1 (Tren Waktu & Musiman):** Dari {total_regulasi:,} peraturan unik bertahun valid, puncaknya terjadi pada **{int(baris_puncak['tahun'])}** dengan **{int(baris_puncak['jumlah_regulasi']):,}** peraturan; **{sejak_2010 / total_regulasi * 100:.1f}%** di antaranya terbit sejak 2010. Bulan berlaku yang paling sering muncul adalah **{bulan_terbanyak}**.

**Insight 2 (Distribusi & Anomali):** Panjang teks sangat timpang: median **{median_teks:,.0f}** karakter, tetapi persentil ke-99 mencapai **{p99_teks:,.0f}**. Sebanyak **{terpotong:,}** potongan mentok di 32.767 karakter (kemungkinan terpotong di sumber). Pencilan jumlah entitas mencakup **{persen_entity_outlier:.1f}%** baris dan referensi silang **{persen_xref_outlier:.1f}%**, artinya ekor panjang adalah ciri data, bukan kasus langka.

**Insight 3 (Korelasi Kunci):** Pasangan variabel dengan korelasi terkuat adalah **{var_a}** dan **{var_b}** (r = **{r_kuat:.2f}**).

**Insight 4 (Perilaku Kategori / Segmen):** Tingkat dengan potongan terbanyak adalah **{terbesar['tingkat']}** ({terbesar['jumlah_pasal']:,} potongan). Skor otoritas rata-rata tertinggi ada pada tingkat **{tertinggi_otoritas['tingkat']}** ({tertinggi_otoritas['rata2_otoritas']:.3f}) dan terendah pada **{terendah_otoritas['tingkat']}** ({terendah_otoritas['rata2_otoritas']:.3f}).

**Insight 5 (Rekomendasi Praktis):** `effective_date` kosong pada **{persen_tanpa_tanggal:.1f}%** baris, `enacting_body` pada **{persen_tanpa_penerbit:.1f}%**, dan `regulation_type` pada **{persen_tanpa_jenis:.1f}%**. Analisis waktu sebaiknya memakai `year` (bukan `effective_date`), dan analisis wilayah perlu dilengkapi tabel rujukan kabupaten/kota ke provinsi agar bisa dipetakan.
"""))

### Insight dalam kata-kata sendiri

*(Isi setelah membaca grafik dan hasil sel di atas. Tulis 5 insight dengan bahasa Anda sendiri, sebutkan angkanya, dan jelaskan maknanya.)*

1. **Insight 1 (Tren Waktu & Musiman):**
2. **Insight 2 (Distribusi & Anomali):**
3. **Insight 3 (Korelasi Kunci):**
4. **Insight 4 (Perilaku Kategori / Segmen):**
5. **Insight 5 (Rekomendasi Praktis):**

## Catatan Milestone 3

- **Unit analisis:** satu baris adalah potongan peraturan, bukan satu peraturan utuh; peraturan unik dihitung dari kombinasi jenis, nomor, tahun, dan penerbit.
- **Normalisasi:** variasi penulisan `regulation_type` (mis. `UU`, `PP`, `KEPPRES`) disatukan pada kolom turunan `jenis_regulasi`; data bersih tidak diubah.
- **Spasial:** data tidak memiliki koordinat, sehingga pola spasial didekati lewat tingkat pemerintahan dan penerbit (`enacting_body`), bukan peta.
- **Winsorization:** `kg_entity_count` dan `kg_cross_ref_count` sudah dipotong pada Milestone 2, sehingga korelasi yang melibatkannya cenderung lebih rendah daripada nilai sebenarnya.
- **Tabel ringkas:** `data/processed/dataset_ringkas.parquet` dibuat oleh notebook ini dan tidak masuk Git.